# OPSD gap on GSM8K

Measures the student vs self-teacher gap, the predictor behind Tufa Labs' OPSD law,
on math instead of code. No training.

**Runtime -> Change runtime type -> GPU (T4).**

In [ ]:
!nvidia-smi

In [ ]:
# repo and dependencies (~3-5 min)
REPO = "https://github.com/kaj04/opsd-gap-gsm8k.git"
!git clone -q $REPO /content/opsd-gap-gsm8k || echo 'already cloned'
%cd /content/opsd-gap-gsm8k
!pip install -q vllm datasets
# pip warns about conflicts with Colab's preinstalled RAPIDS packages. Ignore them.
# Then: Runtime -> Restart session, and run the next cell.

In [ ]:
# run this after every restart: the runtime comes back in /content
%cd /content/opsd-gap-gsm8k
!ls

## A. What GSM8K problems look like

In [ ]:
from datasets import load_dataset
import textwrap

ds = load_dataset("openai/gsm8k", "main", split="test")
print("problems in the test split:", len(ds))

for i in range(3):
    r = ds[i]
    print("=" * 78)
    print("PROBLEM %d" % i)
    print("=" * 78)
    print(textwrap.fill(r["question"], 76))
    print("\n--- reference solution (human written) ---")
    print(r["answer"])
    print()

In [ ]:
# how long the problems are and how many steps they take
import numpy as np

sub = ds.select(range(200))
words = [len(r["question"].split()) for r in sub]
steps = [r["answer"].count("<<") for r in sub]  # <<48/2=24>> marks one calculation
print("words per problem: median %d (min %d, max %d)" % (
    np.median(words), min(words), max(words)))
print("calculations per solution: median %d (min %d, max %d)" % (
    np.median(steps), min(steps), max(steps)))
print("distribution:", {k: steps.count(k) for k in sorted(set(steps))})

## B. What the self-teacher prompts look like

Synthetic examples, no GPU. Check that no privileged context leaks the final answer.

In [ ]:
!python src/inspect_prompts.py

## C. Smoke test

In [ ]:
# 10 problems, 4 samples, 3 conditions (~5 min)
!python src/run_eval.py --n-problems 10 --n-samples 4 --tag smoke \
    --conditions none,feedback_binary,gt_solution

In [ ]:
# the real prompts the model saw during the smoke test
import json

ex = json.load(open("results/smoke_prompt_examples.json", encoding="utf-8"))
for cond, items in ex.items():
    if not items:
        print("(%s: no sample got context)\n" % cond)
        continue
    print("#" * 78)
    print("# CONDITION: %s   (problem %d, sample %d)" % (
        cond, items[0]["problem"], items[0]["sample"]))
    print("#" * 78)
    print(items[0]["prompt"])
    print()

## D. Full run and analysis

In [ ]:
# all conditions (slow - start with 100 problems)
!python src/run_eval.py --n-problems 100 --n-samples 8 --tag run

In [ ]:
# prompts from the full run, one condition at a time
import json

ex = json.load(open("results/run_prompt_examples.json", encoding="utf-8"))
COND = "feedback_diag"  # change to inspect the others
for item in ex[COND]:
    print("-" * 78)
    print(item["prompt"])
    print()

In [ ]:
# table and plot
!python src/analyze.py --tag run
from IPython.display import Image
Image('figures/gaps.png')

In [ ]:
# download the results to commit them back into the repo
from google.colab import files
!zip -qr results.zip results figures
files.download('results.zip')